# 06A - Tokenization and Embeddings Theory

**Guiding question:** how can a legal chatbot turn any contract string into compact, trainable addresses without letting padding become fake evidence?

> **What you already know:** 05B consumes integer token IDs, shifted targets, and real-token masks.
> **The gap this chapter closes:** where those IDs come from, what embedding lookup can and cannot represent, and why masking changes the learning objective.
> **What you'll have by the end:** measured BPE compression, a token-address mental model, and a visible padding failure.
> **What this chapter is not:** a tokenizer benchmark or broad contextual-embedding survey.

We keep Carver & Whitmore LLP and the phrase `non-disclosure agreement`.


In [ ]:
# -- Setup and running legal corpus -----------------------------------------------
from collections import Counter
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

LEGAL_CORPUS = [
    "the non-disclosure agreement protects confidential information",
    "the contract protects confidential records",
    "the agreement lists payment obligations",
    "a non-disclosure clause limits disclosure",
    "the river bank protects the wetland",
    "the bank approves the loan",
]
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})
print(f"Legal examples: {len(LEGAL_CORPUS)}")


## 1 - Character coverage or word efficiency?

```mermaid
flowchart LR
    A["Raw contract"] --> C["Characters: full coverage, long sequence"]
    A --> W["Words: short sequence, unknown-word failures"]
    C --> B["BPE: reusable pieces"]
    W --> B
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** which vocabulary is larger on this corpus: unique characters or unique words? Which one creates the longer sequence?


In [ ]:
# -- Measure the two crude tokenization extremes ---------------------------------
text = " ".join(LEGAL_CORPUS)
characters = list(text)
words = re.findall(r"[a-z-]+", text.lower())
print(f"Character vocabulary: {len(set(characters))}; sequence length: {len(characters)}")
print(f"Word vocabulary:      {len(set(words))}; sequence length: {len(words)}")
print("Takeaway: characters guarantee coverage; words spend fewer positions but break on novelty.")


## 2 - BPE closes the most common seam

The first attempt is deliberately small: words become character tuples with an end marker. Every merge changes only one adjacent pair.

```mermaid
flowchart LR
    A["characters"] --> B["count adjacent pairs"]
    B --> C["merge most frequent pair"]
    C --> D["shorter reusable pieces"]
    D --> B
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Compact BPE mechanics: count, merge, record ---------------------------------
def initialize_words(corpus):
    counts = Counter(re.findall(r"[a-z-]+", " ".join(corpus).lower()))
    return {tuple(word) + ("</w>",): frequency for word, frequency in counts.items()}

def pair_counts(vocabulary):
    counts = Counter()
    for symbols, frequency in vocabulary.items():
        for pair in zip(symbols, symbols[1:]):
            counts[pair] += frequency
    return counts

def merge_pair(vocabulary, pair):
    merged = {}
    for symbols, frequency in vocabulary.items():
        output = []
        index = 0
        while index < len(symbols):
            if index + 1 < len(symbols) and (symbols[index], symbols[index + 1]) == pair:
                output.append(symbols[index] + symbols[index + 1])
                index += 2
            else:
                output.append(symbols[index])
                index += 1
        merged[tuple(output)] = frequency
    return merged

vocabulary = initialize_words(LEGAL_CORPUS)
target_word = tuple("non-disclosure") + ("</w>",)
merge_history = []
target_history = [target_word]

for _ in range(18):
    counts = pair_counts(vocabulary)
    if not counts:
        break
    best_pair, frequency = counts.most_common(1)[0]
    vocabulary = merge_pair(vocabulary, best_pair)
    target_word = next(
        symbols for symbols in vocabulary
        if "".join(symbols).replace("</w>", "") == "non-disclosure"
    )
    merge_history.append((best_pair, frequency))
    target_history.append(target_word)

print(f"Initial target pieces: {len(target_history[0])}")
print(f"After {len(merge_history)} merges: {len(target_history[-1])}")
print(f"Final pieces: {target_history[-1]}")


In [ ]:
# -- Static BPE storyboard: essential evidence without JavaScript ----------------
selected_steps = sorted(set([0, 1, 4, 8, len(target_history) - 1]))
counts = [len(target_history[index]) for index in selected_steps]
labels = [f"merge {index}" for index in selected_steps]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, counts, color="#60a5fa")
ax.set(title="'non-disclosure' compression", ylabel="Piece count")
for bar, index in zip(bars, selected_steps):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        " | ".join(target_history[index]),
        ha="center",
        va="bottom",
        rotation=15,
        color="white",
        fontsize=8,
    )
plt.show()


In [ ]:
# -- BPE animation: every frame comes from recorded merge state ------------------
piece_counts = [len(state) for state in target_history]
fig, ax = plt.subplots(figsize=(9, 4))
ax.set(xlim=(-0.5, len(piece_counts) - 0.5), ylim=(0, max(piece_counts) + 2))
ax.set(xlabel="Merge step", ylabel="Pieces in 'non-disclosure'")
line, = ax.plot([], [], color="#34d399", linewidth=3, marker="o")

def update_bpe(frame):
    visible = frame + 1
    line.set_data(range(visible), piece_counts[:visible])
    ax.set_title(" | ".join(target_history[frame]), color="white")
    return (line,)

bpe_animation = FuncAnimation(fig, update_bpe, frames=len(piece_counts), interval=400, blit=True)
plt.close(fig)
print("Watch common seams close while complete character coverage remains available.")
display(HTML(bpe_animation.to_jshtml()))


### That compressed the string. It still produced symbols, not model inputs.

A vocabulary turns each piece into an address. The address chooses one row; numeric size has no semantic meaning.


## 3 - Token IDs address static rows

```mermaid
flowchart LR
    A["piece: agreement"] --> B["token ID: 17"]
    B --> C["embedding row 17"]
    C --> D["static starting vector"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** before context mixing, should `bank` begin with different lookup vectors in `river bank` and `bank loan`?


In [ ]:
# -- Static lookup versus contextualized token representation --------------------
rng = np.random.default_rng(42)
small_vocab = ["river", "bank", "loan", "wetland", "approves"]
token_to_id = {token: index for index, token in enumerate(small_vocab)}
table = rng.normal(size=(len(small_vocab), 4))

bank_lookup_river = table[token_to_id["bank"]]
bank_lookup_loan = table[token_to_id["bank"]]
river_context = bank_lookup_river + 0.5 * table[token_to_id["river"]]
loan_context = bank_lookup_loan + 0.5 * table[token_to_id["loan"]]

print(f"Static bank rows equal: {np.array_equal(bank_lookup_river, bank_lookup_loan)}")
print(f"After neighbor mixing, contextual rows equal: {np.array_equal(river_context, loan_context)}")
print("Boundary: tokenization supplies addresses; a later model performs context mixing.")


## 4 - Padding creates fake positions unless the objective refuses them

```mermaid
flowchart LR
    A["short sentence"] --> P["pad to batch width"]
    P --> W["wrong: average every cell"]
    P --> M["mask: keep real targets"]
    W --> F["pad frequency distorts loss"]
    M --> C["real-token objective"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style W fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Visible padding failure: one rectangle, two denominators --------------------
token_losses = np.array([
    [0.20, 0.40, 0.30, 0.05, 0.05],
    [0.25, 0.35, 0.05, 0.05, 0.05],
])
real_mask = np.array([
    [1, 1, 1, 0, 0],
    [1, 1, 0, 0, 0],
], dtype=bool)
wrong_mean = token_losses.mean()
correct_mean = token_losses[real_mask].mean()

fig, ax = plt.subplots(figsize=(8, 3))
image = ax.imshow(real_mask, cmap="Blues", vmin=0, vmax=1)
ax.set(xticks=range(5), yticks=range(2), xlabel="Batch position", ylabel="Sequence")
ax.set_title("Real-token mask: blue is evidence, dark is padding")
plt.colorbar(image, ax=ax)
plt.show()

print(f"Wrong all-cell mean: {wrong_mean:.3f}")
print(f"Correct real-token mean: {correct_mean:.3f}")
print("Takeaway: padding changes the objective unless the denominator excludes it.")


## 5 - Production bridge and handoff

GPT-2 uses byte-level BPE: bytes guarantee coverage, learned merges shorten common sequences, and whitespace can be part of a token piece. The lab inspects those real IDs with `tiktoken`.

| Teaching mechanism | Production counterpart | What changes | What does not |
|---|---|---|---|
| character-tuple BPE | byte-level BPE | base symbols and merge vocabulary | frequent pairs become reusable pieces |
| tiny ID map | GPT-2 vocabulary | scale and pretrained artifact | IDs remain addresses |
| static NumPy table | `torch.nn.Embedding` | trainable framework storage | lookup selects rows |
| Boolean real-token mask | `ignore_index` and attention masks | API and mask shape | padding is not evidence |

**Built and measured:** BPE compression, static lookup boundary, padding-loss failure.

**Explained or illustrated:** production byte-level BPE and contextualization boundary.

**Named and out of scope:** unigram tokenizers, multilingual vocabulary design, and causal attention masks.

Next, [06B](06B-tokenization-and-embeddings-pytorch-lab.ipynb) turns every contract into PyTorch code.
